# code for topic modeling of episode annotations and recall transcripts

### imports

In [6]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import hypertools as hyp
from os.path import join as opj
from num2words import num2words
from scipy.signal import resample
from scipy.interpolate import interp1d

### set paths

In [7]:
data_dir = data_dir = r'C:\Users\Alishba\Downloads\prediction files\Transcriptions\data'
annot_dir = opj(data_dir, 'annotations_dfs')
transc_dir = opj(data_dir, 'transcriptions', 'automatic')
ep_traj_dir = opj(data_dir, 'models', 'episodes', 'trajectories')
rec_traj_dir = opj(data_dir, 'models', 'recalls', 'trajectories')
pickle_dir = opj(data_dir, 'pickles')

### load formatted annotations

In [8]:
annot_dir = r'C:\Users\Alishba\Downloads\prediction files\Transcriptions\data\annotations'
atlep1_df = pd.read_csv(opj(annot_dir, 'atlep1.csv'))
atlep2_df = pd.read_csv(opj(annot_dir, 'atlep2.csv'))
arrdev_df = pd.read_csv(opj(annot_dir, 'arrdev.csv'))

### topic modeling parameters

In [9]:
n_topics = 100
episode_wsize = 50
recall_wsize = 200

# vectorizer parameters
vectorizer_params = {
    'model' : 'CountVectorizer', 
    'params' : {
        'stop_words' : 'english'
    }
}

# topic model parameters
semantic_params = {
    'model' : 'LatentDirichletAllocation', 
    'params' : {
        'n_components' : n_topics,
        'learning_method' : 'batch',
        'random_state' : 0,
    }
}

## functions

### for creating episode/recall sliding windows

In [10]:
def format_episode_text(textlist):
    """
    standardize annotation text format for modeling
    """
    formatted = []
    for chunk in textlist:
        lower_nopunc = re.sub("[^\w\s-]+", '', chunk.lower())    # remove all punctuation except for dashes
#         no_acc = lower_nopunc.replace('é', 'e')    # remove accented character
        no_digit = re.sub(r"(\d+)", lambda x: num2words(int(x.group(0))), lower_nopunc)    # convert digits to words
        spaced = ' '.join(no_digit.replace(',', ' ').split())    # deal with inconsistent whitespace
        formatted.append(spaced)
    
    return formatted

<>:7: SyntaxWarning: invalid escape sequence '\w'
<>:7: SyntaxWarning: invalid escape sequence '\w'
C:\Users\Alishba\AppData\Local\Temp\ipykernel_18544\2185898019.py:7: SyntaxWarning: invalid escape sequence '\w'
  lower_nopunc = re.sub("[^\w\s-]+", '', chunk.lower())    # remove all punctuation except for dashes


In [11]:
def get_episode_windows(episode_df, episode_wsize=episode_wsize):
    # throw all annotations into "bag of words"
    episode_bag = format_episode_text(episode_df.loc[:,'Narrative details (external events)':'Setting'].apply(
        lambda x: ', '.join(x.fillna('')), axis=1).values.tolist())
    # create sliding windows
    episode_w = []
    for idx, sentence in enumerate(episode_bag):
        episode_w.append(' '.join(episode_bag[idx:idx+episode_wsize]))

    return episode_w

In [12]:
def get_recall_windows(transcript, recall_wsize=recall_wsize):
    rec_list = transcript.split()
    # create sliding windows
    recall_w = []
    for ix, word in enumerate(rec_list):
        recall_w.append(' '.join(rec_list[ix:ix+recall_wsize]))
        
    return recall_w

### for interpolating episode trajectories

In [13]:
def find_midpoint_time(df):
    """
    returns list of timepoints at middle of each annotation segment
    """
    # use timestamp of last frame from episode to find midpoint of last annotation
    df_shapes = [atlep1_df.shape, atlep2_df.shape, arrdev_df.shape]
    endframe_times = [1466.0, 1316.52, 1236.6]
    endframe_time = endframe_times[df_shapes.index(df.shape)]

    midpoint_times = []
    for i, tpt in enumerate(df['Onset time']):
        try:
            midpoint_times.append((tpt + df['Onset time'][i+1]) / 2)
        except KeyError:    # handle last annotation
            midpoint_times.append((tpt + endframe_time) / 2)
                    
    return midpoint_times, endframe_time

In [14]:
def interpolate_episode(traj, df, resolution=1):
    """
    uses linear interpolation to resample episode trajectory timeseries to desired resolution. 
    'resolution' is in units of seconds.
    """
    # get middle timepoint for each annotation
    midpoint_times, endframe_time = find_midpoint_time(df)
    new_traj = np.arange(int(round(endframe_time)), step=resolution)
    interp_func = interp1d(midpoint_times, traj, axis=0, fill_value='extrapolate')
    
    return interp_func(new_traj)

## main topic modeling function

In [15]:
def fit_and_transform(documents, vec_params=vectorizer_params, sem_params=semantic_params, 
                        corpus=None, resample_shape=None, return_windows=False):
    # handle annotations
    if isinstance(documents, pd.DataFrame):
        windows = get_episode_windows(documents, episode_wsize)
        corpus = windows if not corpus else corpus
        # fit topic model and transform documents
        traj =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
        
        # interpolate to length of episode (seconds)
        if return_windows:
            return interpolate_episode(traj, documents), windows
        else:
            return interpolate_episode(traj, documents)
        
    # handle recall transcripts
    elif isinstance(documents, str):
        if not corpus:
            raise ValueError("You must pass a training corpus to transform recall transcripts")
        windows = get_recall_windows(documents, recall_wsize)
        traj =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
        
        # resample to corresponding episode length
        return resample(traj, resample_shape) 

## model episode content, get sliding windows for fitting recall models

In [16]:
atlep1_traj, atlep1_windows = fit_and_transform(atlep1_df, return_windows=True)
atlep2_traj, atlep2_windows = fit_and_transform(atlep2_df, return_windows=True)
arrdev_traj, arrdev_windows = fit_and_transform(arrdev_df, return_windows=True)

## save episode trajectories

In [17]:
np.save(f'{ep_traj_dir}/atlep1_trajectory', atlep1_traj)
np.save(f'{ep_traj_dir}/atlep2_trajectory', atlep2_traj)
np.save(f'{ep_traj_dir}/arrdev_trajectory', arrdev_traj)

## load in and model recall transcripts

In [18]:
recall_trajectories = {
    'atlep1' : [],
    'prediction' : [],
    'delayed' : [],
    'atlep2' : [],
    'arrdev' : []
}

total = sum([len([f for f in files if f.endswith('corrected.wav.txt')]) for r, d, files in os.walk(transc_dir)])
# walk transcription directory structure
currfile = 1
for root, dirs, files in os.walk(transc_dir):
    
    # FOR USE WITH AUTOMATIC TRANSCRIPTS -- REMOVE WHEN SWITCHING TO MANUAL
    transcripts = [f for f in files if f.endswith('corrected.wav.txt')]
    for transc in transcripts:

        # assign correct episode windows, corresponding episode trajectory shape, and dict key
        if any('prediction' in t for t in transcripts) or 'delayed' in transc:
            corpus = atlep1_windows
            resample_shape = atlep1_traj.shape[0]
            if 'recall' in transc:
                rectype = 'atlep1'
            elif 'prediction' in transc:
                rectype = 'prediction'
            elif 'delayed' in transc:
                rectype = 'delayed'
            else:
                raise ValueError('Transcript is not a recognized option')
            
        elif '-A-' in root:
            corpus = atlep2_windows
            resample_shape = atlep2_traj.shape[0]
            rectype = 'atlep2'
            
        else:
            corpus = arrdev_windows
            resample_shape = arrdev_traj.shape[0]
            rectype = 'arrdev'
            
        with open(opj(root ,transc), 'r') as f:
            # FOR USE WITH AUTOMATIC TRANSCRIPTS -- REMOVE WHEN SWITCHING TO MANUAL
            transcript = ' '.join([line.split(',')[0].lower() for line in f.read().split('\n')])
            
        # fit topic model to episode annotations, 
        print(f'modeling transcript {currfile}/{total}...    {transc}')
        p_traj = fit_and_transform(transcript, resample_shape=resample_shape, corpus=corpus)
        
        recall_trajectories[rectype].append((transc.split('-')[0],p_traj))
        currfile += 1

modeling transcript 1/228...    debug3Z74G_debug6CQ6D-delayed-corrected.wav.txt
modeling transcript 2/228...    debug3Z74G_debug6CQ6D-recall-corrected.wav.txt
modeling transcript 3/228...    debug5ytRS_debug22nvn-prediction-corrected.wav.txt
modeling transcript 4/228...    debug5ytRS_debug22nvn-recall-corrected.wav.txt
modeling transcript 5/228...    debug0GSk9_debugttOEY-delayed-corrected.wav.txt
modeling transcript 6/228...    debug0GSk9_debugttOEY-recall-corrected.wav.txt
modeling transcript 7/228...    debuguhHBa_debug1HOCZ-prediction-corrected.wav.txt
modeling transcript 8/228...    debuguhHBa_debug1HOCZ-recall-corrected.wav.txt
modeling transcript 9/228...    debug9ngWi_debugpMjGl-delayed-corrected.wav.txt
modeling transcript 10/228...    debug9ngWi_debugpMjGl-recall-corrected.wav.txt
modeling transcript 11/228...    debugfGFHT_debugLCOgS-prediction-corrected.wav.txt
modeling transcript 12/228...    debugfGFHT_debugLCOgS-recall-corrected.wav.txt
modeling transcript 13/228...    d

## save individual trajectories

In [19]:
for rectype, data in recall_trajectories.items():
    for (turkid, traj) in data:
        np.save(opj(rec_traj_dir, rectype, f'{turkid}.npy'), traj)

## create and save average recall trajectories

In [20]:
for rectype, data in recall_trajectories.items():
    avg_trajectory = np.array([traj for (turkid, traj) in data]).mean(axis=0)
    np.save(opj(rec_traj_dir, rectype, 'avg_trajectory.npy'), avg_trajectory)